# Hands-on Exercise 2: Extract innovation-culture triples with RAG

This notebook is a small teaching demo inspired by **Li, Mai, Shen, Yang & Zhang (2026), _Dissecting Corporate Culture Using Generative AI_**.

We use a **made-up, already segmented analyst report** so that the logic is transparent:

1. Start with a focal culture segment.
2. Apply a Chain of Thoughts prompt to extract a tripple.
3. If the focal segment lacks enough evidence, mark **“I need more context.”**
4. Retrieve related segments from the same report using a RAG-style procedure.
5. Re-run extraction using the focal segment plus retrieved context.
6. Produce culture triples with tone and canonical categories.

The goal is not to reproduce the paper. The goal is to show students how a research variable can be built from text.

## Learning goals

By the end, students should understand:

- why a single culture sentence may not be enough to identify causes and effects;
- how RAG can add relevant context from the same report;
- how a structured prompt differs from a generic “summarise this” prompt;
- why canonicalisation and human inputs are needed before using triples in regressions.

## protocol used in this demo

The paper's analyst-report workflow can be simplified as:

**Step 1: Culture segment identification**  
Identify segments that substantively discuss corporate or organisational culture.

**Step 2: First-pass extraction**  
For each culture-related segment, ask the model to identify:

- specific corporate culture;
- culture type;
- whether there is detailed causal analysis;
- causes of the culture;
- outcomes from the culture;
- tone;
- cause-effect graph triples.

**Step 3: RAG only when needed**  
If the model says **“I need more context”**, retrieve related segments from the same analyst report.  
The paper retrieves similar segments, filters them using culture-relevance probabilities, adds adjacent segments, and uses skip markers for non-adjacent text.

**Step 4: Rerun extraction**  
Run the same extraction again with the augmented context.

**Step 5: Canonicalise**  
Map raw causes and effects to standard categories so the output can be aggregated.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

## 1. Create a pseudo segmented analyst report

This is a fictional analyst report on **NovaCloud Inc.**

Some segments are about culture; some are not.  
The focal segment is **A04**, which mentions an *experimentation culture* but does not fully explain its causes or outcomes by itself.

In [2]:
segments = [
    {
        "report_id": "NovaCloud_Analyst_Report_2026",
        "segment_id": "A01",
        "segment_order": 1,
        "text": "NovaCloud reported 18% revenue growth, above our 14% estimate, driven by stronger enterprise demand and better cloud utilisation.",
        "culture_probability": 0.10,
        "role_for_demo": "not culture"
    },
    {
        "report_id": "NovaCloud_Analyst_Report_2026",
        "segment_id": "A02",
        "segment_order": 2,
        "text": "Since the new CEO took over, NovaCloud shifted from a centralised AI product-cycle process to small autonomous product squads with direct customer feedback loops, a design intended to spread experimentation beyond the founder-led engineering team.",
        "culture_probability": 0.92,
        "role_for_demo": "RAG context: cause"
    },
    {
        "report_id": "NovaCloud_Analyst_Report_2026",
        "segment_id": "A03",
        "segment_order": 3,
        "text": "Management gives senior engineers protected time to test experimental AI ideas and requires teams to share post-mortems when product projects fail.",
        "culture_probability": 0.90,
        "role_for_demo": "RAG context: cause"
    },
    {
        "report_id": "NovaCloud_Analyst_Report_2026",
        "segment_id": "A04",
        "segment_order": 4,
        "text": "We view NovaCloud's ability to sustain its AI product cycle as tied to whether this experimentation culture can scale beyond the founder-led engineering team.",
        "culture_probability": 0.97,
        "role_for_demo": "focal culture segment"
    },
    {
        "report_id": "NovaCloud_Analyst_Report_2026",
        "segment_id": "A05",
        "segment_order": 5,
        "text": "Where product squads have adopted this experimentation cadence, release cycles have shortened from quarterly to monthly and net revenue retention improved from 116% to 124% as customers adopted more AI modules.",
        "culture_probability": 0.89,
        "role_for_demo": "RAG context: effect"
    },
    {
        "report_id": "NovaCloud_Analyst_Report_2026",
        "segment_id": "A06",
        "segment_order": 6,
        "text": "Our price target is based on 9.5x next-twelve-month revenue, a discount to high-growth software peers.",
        "culture_probability": 0.08,
        "role_for_demo": "not culture / valuation"
    },
]

df = pd.DataFrame(segments)
df

,report_id,segment_id,segment_order,text,culture_probability,role_for_demo
0,NovaCloud_Analyst_Report_2026,A01,1,"NovaCloud reported 18% revenue growth, above our 14% estimate, driven by stronger enterprise demand and better cloud...",0.10,not culture
1,NovaCloud_Analyst_Report_2026,A02,2,"Since the new CEO took over, NovaCloud shifted from a centralised AI product-cycle process to small autonomous produ...",0.92,RAG context: cause
2,NovaCloud_Analyst_Report_2026,A03,3,Management gives senior engineers protected time to test experimental AI ideas and requires teams to share post-mort...,0.90,RAG context: cause
3,NovaCloud_Analyst_Report_2026,A04,4,We view NovaCloud's ability to sustain its AI product cycle as tied to whether this experimentation culture can scal...,0.97,focal culture segment
4,NovaCloud_Analyst_Report_2026,A05,5,"Where product squads have adopted this experimentation cadence, release cycles have shortened from quarterly to mont...",0.89,RAG context: effect
5,NovaCloud_Analyst_Report_2026,A06,6,"Our price target is based on 9.5x next-twelve-month revenue, a discount to high-growth software peers.",0.08,not culture / valuation


## 2. The triple extraction prompt

In the actual paper, the model is asked to proceed step by step and return JSON.

For teaching, the prompt below is **adapted and shortened**. It keeps the same logic:

- identify the specific culture;
- classify the culture type;
- decide whether detailed causal analysis is present;
- extract causes, outcomes, tone, and triples;
- allow **“I need more context”** when the segment alone is insufficient.

In [3]:
LI_STYLE_EXTRACTION_PROMPT = """
As an expert specializing in corporate culture and causal reasoning, analyze the input segment from a sell-side analyst report.

Step-by-step tasks:
1. Identify the specific corporate culture discussed.
2. Classify it into one culture type:
   - Collaboration and people-focused
   - Customer-oriented
   - Innovation and adaptability
   - Integrity and risk management
   - Performance-oriented
   - Miscellaneous
3. Decide whether the segment contains detailed causal analysis.
   If more context is needed, output "I need more context".
4. Extract explicitly mentioned causes of the culture.
5. Extract explicitly mentioned outcomes from the culture.
6. Determine tone: positive, negative, or neutral.
7. Extract cause-effect graph triples.
   Each triple must include the corporate culture as one entity.
   The other entity must be either a cause of the culture or an outcome from the culture.

Return JSON only.
"""

print(LI_STYLE_EXTRACTION_PROMPT)


As an expert specializing in corporate culture and causal reasoning, analyze the input segment from a sell-side analyst report.

Step-by-step tasks:
1. Identify the specific corporate culture discussed.
2. Classify it into one culture type:
   - Collaboration and people-focused
   - Customer-oriented
   - Innovation and adaptability
   - Integrity and risk management
   - Performance-oriented
   - Miscellaneous
3. Decide whether the segment contains detailed causal analysis.
   If more context is needed, output "I need more context".
4. Extract explicitly mentioned causes of the culture.
5. Extract explicitly mentioned outcomes from the culture.
6. Determine tone: positive, negative, or neutral.
7. Extract cause-effect graph triples.
   Each triple must include the corporate culture as one entity.
   The other entity must be either a cause of the culture or an outcome from the culture.

Return JSON only.



## 3. First pass: focal segment only

We now mimic the first-pass LLM extraction on **A04**.

The focal segment clearly contains a culture phrase: *experimentation culture*.  
But by itself, it does not fully explain:

- what caused this culture; or
- what business outcomes it produces.

So the correct first-pass response is **“I need more context.”**

In [4]:
focal_id = "A04"
focal = df.loc[df["segment_id"].eq(focal_id)].iloc[0]

def first_pass_extraction(segment_row):
    # Deterministic teaching stub that mimics the expected output of a Li-style first-pass LLM extraction.
    text = segment_row["text"].lower()

    if "experimentation culture" in text:
        return {
            "input_id": segment_row["segment_id"],
            "identified_corporate_culture": "experimentation culture",
            "corporate_culture_type": "Innovation and adaptability",
            "detailed_causal_analysis": "I need more context",
            "causes_of_culture": [],
            "outcomes_from_culture": [],
            "tone": "positive",
            "causal_graph_triples": [],
            "explanation": "The focal segment identifies experimentation culture and links it to sustaining the AI product cycle, but does not fully identify causes or outcomes."
        }

    return {
        "input_id": segment_row["segment_id"],
        "identified_corporate_culture": "N/A",
        "corporate_culture_type": "N/A",
        "detailed_causal_analysis": "No",
        "causes_of_culture": [],
        "outcomes_from_culture": [],
        "tone": "neutral",
        "causal_graph_triples": [],
        "explanation": "No substantive corporate culture discussion."
    }

round1 = first_pass_extraction(focal)
print(json.dumps(round1, indent=2))

{
  "input_id": "A04",
  "identified_corporate_culture": "experimentation culture",
  "corporate_culture_type": "Innovation and adaptability",
  "detailed_causal_analysis": "I need more context",
  "causes_of_culture": [],
  "outcomes_from_culture": [],
  "tone": "positive",
  "causal_graph_triples": [],
  "explanation": "The focal segment identifies experimentation culture and links it to sustaining the AI product cycle, but does not fully identify causes or outcomes."
}


## 4. RAG: retrieve context from the same report

The RAG procedure here follows the spirit of the paper:

- restrict retrieval to the same report;
- compute semantic similarity between the focal segment and other segments;
- retrieve up to five similar segments;
- filter by culture-relevance probability;
- add immediately adjacent segments;
- order the final context by report position;
- insert skip markers when non-adjacent segments are combined.

For a small classroom example, TF-IDF cosine similarity is enough to make the mechanics visible.

In [5]:
def retrieve_rag_context(df, focal_id, top_k=5, percentile_threshold=75, include_adjacent=True):
    """Retrieve Li-style RAG context for a focal segment.

    Teaching simplification:
    - TF-IDF cosine similarity approximates semantic similarity.
    - culture_probability approximates the paper's BERT culture-relevance probability.
    """
    focal_row = df.loc[df["segment_id"].eq(focal_id)].iloc[0]
    same_report = df[df["report_id"].eq(focal_row["report_id"])].copy()

    # Similarity
    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
    X = vectorizer.fit_transform(same_report["text"])
    focal_idx = same_report.index.get_loc(focal_row.name)
    sims = cosine_similarity(X[focal_idx], X).ravel()
    same_report["similarity_to_focal"] = sims

    # Paper-style culture-probability filter
    prob_cutoff = np.percentile(same_report["culture_probability"], percentile_threshold)
    candidates = same_report[
        (same_report["segment_id"] != focal_id)
        & (same_report["culture_probability"] >= prob_cutoff)
    ].copy()

    semantic_top = (
        candidates
        .sort_values("similarity_to_focal", ascending=False)
        .head(top_k)
        .assign(retrieval_reason="semantic_top_k_after_culture_filter")
    )

    selected = semantic_top.copy()

    if include_adjacent:
        focal_order = int(focal_row["segment_order"])
        adjacent = same_report[
            same_report["segment_order"].isin([focal_order - 1, focal_order + 1])
        ].copy()
        adjacent = adjacent[adjacent["segment_id"] != focal_id]
        adjacent["retrieval_reason"] = "adjacent_segment"
        selected = pd.concat([selected, adjacent], ignore_index=True)

    # If a segment is selected for two reasons, combine the reasons.
    selected = (
        selected
        .groupby(["report_id", "segment_id", "segment_order", "text", "culture_probability", "role_for_demo"], as_index=False)
        .agg({
            "similarity_to_focal": "max",
            "retrieval_reason": lambda x: " + ".join(sorted(set(x)))
        })
        .sort_values("segment_order")
        .reset_index(drop=True)
    )

    return selected, prob_cutoff

rag_segments, prob_cutoff = retrieve_rag_context(df, focal_id)

print(f"75th percentile culture-probability cutoff: {prob_cutoff:.3f}")
rag_segments[["segment_id", "segment_order", "retrieval_reason", "similarity_to_focal", "culture_probability", "text"]]

75th percentile culture-probability cutoff: 0.915


,segment_id,segment_order,retrieval_reason,similarity_to_focal,culture_probability,text
0,A02,2,semantic_top_k_after_culture_filter,0.270012,0.92,"Since the new CEO took over, NovaCloud shifted from a centralised AI product-cycle process to small autonomous produ..."
1,A03,3,adjacent_segment,0.025072,0.90,Management gives senior engineers protected time to test experimental AI ideas and requires teams to share post-mort...
2,A05,5,adjacent_segment,0.039687,0.89,"Where product squads have adopted this experimentation cadence, release cycles have shortened from quarterly to mont..."


### Teaching interpretation

The focal segment A04 tells us there is an **experimentation culture**.

RAG adds:

- **A02**: possible cause — CEO-led shift to autonomous product squads and customer feedback loops;
- **A03**: possible cause — protected experimentation time and post-mortems;
- **A05**: possible effect — faster release cycles and higher net revenue retention.

## 5. Assemble the final text passed to the extraction model

This cell creates a compact version of the final input:

- focal segment;
- retrieved context segments;
- skip markers if selected context is non-adjacent.

This mirrors the idea that the LLM should see only relevant context, not the whole report.

In [6]:
def assemble_rag_input(df, focal_id, rag_segments):
    focal_row = df.loc[df["segment_id"].eq(focal_id)].iloc[0]
    selected = rag_segments.copy().sort_values("segment_order")

    lines = []
    lines.append(f"FOCAL {focal_row['segment_id']}: {focal_row['text']}")
    lines.append("")
    lines.append("ADDITIONAL CONTEXT:")

    prev_order = None
    for _, row in selected.iterrows():
        if prev_order is not None and int(row["segment_order"]) > prev_order + 1:
            lines.append("[...]")
        lines.append(f"{row['segment_id']}: {row['text']}")
        prev_order = int(row["segment_order"])

    return "\n".join(lines)

final_context = assemble_rag_input(df, focal_id, rag_segments)
print(final_context)

FOCAL A04: We view NovaCloud's ability to sustain its AI product cycle as tied to whether this experimentation culture can scale beyond the founder-led engineering team.

ADDITIONAL CONTEXT:
A02: Since the new CEO took over, NovaCloud shifted from a centralised AI product-cycle process to small autonomous product squads with direct customer feedback loops, a design intended to spread experimentation beyond the founder-led engineering team.
A03: Management gives senior engineers protected time to test experimental AI ideas and requires teams to share post-mortems when product projects fail.
[...]
A05: Where product squads have adopted this experimentation cadence, release cycles have shortened from quarterly to monthly and net revenue retention improved from 116% to 124% as customers adopted more AI modules.


## 6. Rerun extraction with RAG context

The deterministic function below mimics the expected output of a structured LLM extraction after RAG context is added.

This is the key classroom point:

> RAG supplies the missing evidence needed for the structured extraction.

In [7]:
def rag_augmented_extraction(focal_row, rag_segments):
    # Deterministic teaching stub for the expected RAG-augmented extraction.
    context_text = " ".join(rag_segments["text"].tolist())

    triples = []
    causes = []
    effects = []

    if "new CEO" in context_text and "product squads" in context_text:
        causes.append({
            "raw_phrase": "new CEO's shift to autonomous product squads",
            "canonical_entity": "Management team / Business strategy",
            "evidence_segment": "A02",
            "evidence": "Since the new CEO took over, NovaCloud shifted ... to small autonomous product squads"
        })
        triples.append({
            "source_entity": "new CEO's shift to autonomous product squads",
            "relation": "fosters",
            "target_entity": "experimental innovation culture",
            "tone": "positive",
            "evidence_segment": "A02/A04"
        })

    if "protected time" in context_text and "post-mortems" in context_text:
        causes.append({
            "raw_phrase": "protected experimentation time and failure post-mortems",
            "canonical_entity": "Business strategy / Employee practices",
            "evidence_segment": "A03",
            "evidence": "protected time to test experimental AI ideas ... share post-mortems"
        })
        triples.append({
            "source_entity": "protected experimentation time and failure post-mortems",
            "relation": "reinforce",
            "target_entity": "experimental innovation culture",
            "tone": "positive",
            "evidence_segment": "A03/A04"
        })

    if "release cycles have shortened" in context_text and "net revenue retention improved" in context_text:
        effects.append({
            "raw_phrase": "shorter release cycles and higher net revenue retention",
            "canonical_entity": "Innovation / Market share and growth",
            "evidence_segment": "A05",
            "evidence": "release cycles have shortened ... net revenue retention improved from 116% to 124%"
        })
        triples.append({
            "source_entity": "experimental innovation culture",
            "relation": "supports",
            "target_entity": "shorter release cycles and higher net revenue retention",
            "tone": "positive",
            "evidence_segment": "A04/A05"
        })

    return {
        "input_id": focal_row["segment_id"],
        "identified_corporate_culture": "experimental innovation culture",
        "corporate_culture_type": "Innovation and adaptability",
        "detailed_causal_analysis": "Yes",
        "causes_of_culture": causes,
        "outcomes_from_culture": effects,
        "tone": "positive",
        "causal_graph_triples": triples,
        "explanation": "The focal segment identifies experimentation culture; retrieved context provides causes and outcomes."
    }

round2 = rag_augmented_extraction(focal, rag_segments)
print(json.dumps(round2, indent=2))

{
  "input_id": "A04",
  "identified_corporate_culture": "experimental innovation culture",
  "corporate_culture_type": "Innovation and adaptability",
  "detailed_causal_analysis": "Yes",
  "causes_of_culture": [
    {
      "raw_phrase": "new CEO's shift to autonomous product squads",
      "canonical_entity": "Management team / Business strategy",
      "evidence_segment": "A02",
      "evidence": "Since the new CEO took over, NovaCloud shifted ... to small autonomous product squads"
    },
    {
      "raw_phrase": "protected experimentation time and failure post-mortems",
      "canonical_entity": "Business strategy / Employee practices",
      "evidence_segment": "A03",
      "evidence": "protected time to test experimental AI ideas ... share post-mortems"
    }
  ],
  "outcomes_from_culture": [
    {
      "raw_phrase": "shorter release cycles and higher net revenue retention",
      "canonical_entity": "Innovation / Market share and growth",
      "evidence_segment": "A05",
    

## 7. Convert extraction output into a triple table

This is the form students can recognise as a research-data output.

Each row is one triple:

**entity 1 → relation → entity 2**

The canonical version maps raw phrases to categories that can be counted across many reports.

In [8]:
triple_rows = []
for t in round2["causal_graph_triples"]:
    source = t["source_entity"]
    target = t["target_entity"]

    if target == "experimental innovation culture":
        canonical = "Management team / Business strategy / Employee practices → Innovation and Adaptability"
        direction = "cause_to_culture"
    else:
        canonical = "Innovation and Adaptability → Innovation / Market share and growth"
        direction = "culture_to_effect"

    triple_rows.append({
        "focal_segment": round2["input_id"],
        "triple": f"{source} → {t['relation']} → {target}",
        "tone": t["tone"].title(),
        "direction": direction,
        "canonical_version": canonical,
        "evidence_segments": t["evidence_segment"],
    })

triples_df = pd.DataFrame(triple_rows)
triples_df

,focal_segment,triple,tone,direction,canonical_version,evidence_segments
0,A04,new CEO's shift to autonomous product squads → fosters → experimental innovation culture,Positive,cause_to_culture,Management team / Business strategy / Employee practices → Innovation and Adaptability,A02/A04
1,A04,protected experimentation time and failure post-mortems → reinforce → experimental innovation culture,Positive,cause_to_culture,Management team / Business strategy / Employee practices → Innovation and Adaptability,A03/A04
2,A04,experimental innovation culture → supports → shorter release cycles and higher net revenue retention,Positive,culture_to_effect,Innovation and Adaptability → Innovation / Market share and growth,A04/A05


## 8. Show final combined evidence for each triple

This table is useful for  validation.

A student should be able to check whether each triple is supported by the focal segment plus retrieved context.

In [9]:
def segment_text(segment_ids):
    ids = []
    for part in segment_ids.split("/"):
        ids.append(part.strip())
    return "\n".join(
        f"{sid}: {df.loc[df['segment_id'].eq(sid), 'text'].iloc[0]}"
        for sid in ids
        if sid in set(df["segment_id"])
    )

evidence_df = triples_df.copy()
evidence_df["combined_evidence_text"] = evidence_df["evidence_segments"].apply(segment_text)
evidence_df[["triple", "tone", "canonical_version", "combined_evidence_text"]]

,triple,tone,canonical_version,combined_evidence_text
0,new CEO's shift to autonomous product squads → fosters → experimental innovation culture,Positive,Management team / Business strategy / Employee practices → Innovation and Adaptability,"A02: Since the new CEO took over, NovaCloud shifted from a centralised AI product-cycle process to small autonomous ..."
1,protected experimentation time and failure post-mortems → reinforce → experimental innovation culture,Positive,Management team / Business strategy / Employee practices → Innovation and Adaptability,A03: Management gives senior engineers protected time to test experimental AI ideas and requires teams to share post...
2,experimental innovation culture → supports → shorter release cycles and higher net revenue retention,Positive,Innovation and Adaptability → Innovation / Market share and growth,A04: We view NovaCloud's ability to sustain its AI product cycle as tied to whether this experimentation culture can...


## 9. Save teaching outputs

The notebook saves three simple files:

- `005_pseudo_report_segments.csv`
- `005_pseudo_report_rag_context.csv`
- `005_pseudo_report_triples.csv`

These can be used in slides or as handouts.

In [10]:
OUTPUT_DIR = Path(".")
df.to_csv(OUTPUT_DIR / "005_pseudo_report_segments.csv", index=False)
rag_segments.to_csv(OUTPUT_DIR / "005_pseudo_report_rag_context.csv", index=False)
triples_df.to_csv(OUTPUT_DIR / "005_pseudo_report_triples.csv", index=False)

print("Saved:")
print(OUTPUT_DIR / "005_pseudo_report_segments.csv")
print(OUTPUT_DIR / "005_pseudo_report_rag_context.csv")
print(OUTPUT_DIR / "005_pseudo_report_triples.csv")

Saved:
005_pseudo_report_segments.csv
005_pseudo_report_rag_context.csv
005_pseudo_report_triples.csv


## 10. Class discussion

Use the pseudo report to ask:

1. Why is A04 insufficient by itself?
2. Which retrieved segments provide causes?
3. Which retrieved segment provides outcomes?
4. Are the extracted triples directly supported by evidence?
5. How would you aggregate these triples to the report level, firm-quarter level, or firm-year level?
